# Forecasting France's Solar Electricity Production
### Time Series Analysis Project — ETS, SARIMA and Prophet Decomposition

**Team:** Enzo Augié, Etienne Girard

**Data:** ODRÉ / RTE — *Monthly regional electricity production by source* (data.gouv.fr), January 2014 → December 2025.

---

**Contents**

1. Introduction
2. Data loading and preparation
3. Exploratory analysis
4. Decomposition of the series
5. Evaluation protocol and baseline
6. ETS model (exponential smoothing)
7. SARIMA model
8. Prophet model
9. Methodological comparison of forecasts
10. 2026 forecast
11. Business recommendations
12. Conclusion and limitations
13. References
14. Course glossary

## 1. Introduction

### Context
France has committed to a rapid scale-up of renewable energy. Among these sources, **solar photovoltaic** is the fastest-growing one: its annual production has more than quintupled between 2014 and 2025. Unlike nuclear or dam-based hydro, solar is a **non-dispatchable** ("must-run") source of production: you cannot choose when it arrives, it depends on solar radiation.

This characteristic raises concrete questions for several stakeholders:

- **The grid operator (RTE)** must anticipate the supply/demand balance, plan export capacity and flexibility needs (storage, nuclear load-following).
- **Producers and aggregators** need to anticipate their volumes for sales contracts and hedging on forward markets.
- **EDF and nuclear plant operators** schedule reactor maintenance outages during periods when solar production is high and demand is low.
- **Public authorities** track the actual trajectory against the targets of the multi-year energy plan (PPE).

### Research question
> **What monthly solar production can France expect over the next 12 months, with what uncertainty, and which method (ETS, SARIMA or Prophet) provides the most reliable forecast?**

### Why this series?
The dataset covers 6 energy sources × 13 regions. We chose **national solar production** (sum of the 13 metropolitan regions) because it simultaneously shows:
- a strong, nonlinear **trend** (fleet deployment);
- a very pronounced **seasonality** (sunshine) whose amplitude grows with the level, hence **multiplicative**;
- a current and easily communicated business stake.

It is a textbook case for comparing the three families of methods required by the project. Section 3 justifies this choice against the other energy sources.

## 2. Data loading and preparation

In [ ]:
import warnings, logging, itertools
warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
logging.getLogger("prophet").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from prophet import Prophet
logging.getLogger("cmdstanpy").disabled = True   # Prophet re-enables this logger on import
warnings.simplefilter("ignore")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 4.5)
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

COLORS = {"Actual": "black", "Seasonal naive": "grey", "ETS": "tab:blue",
          "SARIMA": "tab:orange", "Prophet": "tab:green"}

In [ ]:
# The file is ~20 MB because of the geographic columns (region polygons):
# we only load the first 9 columns.
raw = pd.read_csv("data/production-regionale-mensuelle-filiere.csv", sep=";", usecols=range(9))
raw.columns = ["month", "insee_code", "region", "nuclear", "thermal",
               "hydro", "wind", "solar", "bioenergy"]
raw["month"] = pd.to_datetime(raw["month"], format="%Y-%m")
SOURCES = ["nuclear", "thermal", "hydro", "wind", "solar", "bioenergy"]

print("Shape      :", raw.shape)
print("Period     :", raw["month"].min().date(), "→", raw["month"].max().date(),
      f"({raw['month'].nunique()} months)")
print("Regions    :", raw["region"].nunique())
raw.head()

In [ ]:
# Data quality checks
print("Missing values per column:")
print(raw.isna().sum(), "\n")
print("Regions with no nuclear value:")
print(sorted(raw.loc[raw["nuclear"].isna(), "region"].unique()), "\n")
print("Duplicates (month, region):", raw.duplicated(["month", "region"]).sum())
print("Observations per region:", raw.groupby("region").size().unique())
print("Negative values:", (raw[SOURCES] < 0).sum().sum())

**Interpretation.**

- *Technical.* The panel is **complete and balanced**: 144 months × 13 regions = 1,872 rows, with no duplicates. The only missing values are in the nuclear column.
- *Business.* These missing values are not data-entry errors: they correspond exactly to the **6 regions that host no nuclear plant** (Brittany, Île-de-France, PACA, Pays de la Loire, Bourgogne-Franche-Comté, Corsica). A missing value there therefore means "zero production." It can be replaced by 0 without introducing bias.
- *Negative values (34).* They are rare and small: nuclear in Occitanie (6 months, down to −50 GWh), thermal in 4 regions (a few GWh), and wind in Corsica (≈ 0). This is most likely **net** production: when a plant is shut down, the consumption of its auxiliary systems exceeds its output. These are therefore genuine values, not errors, and they do not affect solar.

  > **Hypothesis verified (partially).** We found no explicit textual confirmation — neither in the dataset's description page on data.gouv.fr, nor in SDES's methodological note on photovoltaics — that these negative values correspond to auxiliary-system consumption exceeding the output of an idle plant. The explanation remains **physically consistent and well documented for the power sector in general** (a plant "on standby" or under continuous maintenance keeps consuming electricity for its own needs — cooling, instrumentation — without producing any), and it is consistent with the profile observed here (nuclear and thermal only, never solar, small and occasional amounts). We therefore treat this explanation as the **most likely hypothesis, but not formally confirmed by the dataset's documentation**; a definitive confirmation would require contacting ODRÉ support or consulting RTE's technical documentation on éCO2mix aggregation.
- No imputation or correction is needed for solar, our variable of interest.

> **Hypothesis verified — "provisional" data.** The dataset's description page on data.gouv.fr (last updated: **25 February 2026**) confirms that a revision has already taken place: *"recalculation of photovoltaic self-consumption,"* which **raised 2022-2024 solar production** relative to earlier publications. This is a direct, documented precedent: this very dataset has already been revised after the fact for solar, precisely on the question of self-consumption.
>
> This point lines up with a discrepancy we quantified by cross-checking our own figures against two official publications: national solar production for 2025 computed from **this dataset** is **32.9 TWh** (§10), a figure very close to that of [RTE's 2025 Electricity Report](https://assets.rte-france.com/prod/public/2026-02/Bilan-electrique-2025-principaux-resultats.pdf) (≈ 32 TWh, +33%, which measures production **fed into the grid**). The [SDES Q4 2025 solar photovoltaic dashboard](https://www.statistiques.developpement-durable.gouv.fr/tableau-de-bord-solaire-photovoltaique-quatrieme-trimestre-2025), on the other hand, reports **37.0 TWh (+46%)**, a higher figure because it **includes an estimate of self-consumption** (confirmed by [SDES's methodological note](https://www.statistiques.developpement-durable.gouv.fr/sites/default/files/2022-02/methodologie_pv_fevrier2022.pdf): *"Only the electricity sold to the grid is precisely known,"* self-consumption being estimated through a separate statistical model).
>
> **Conclusion:** our series (like RTE's) measures production **fed into the grid**, not total production. The roughly 4 TWh (11%) gap with the SDES figure is therefore not a simple consolidation lag, but a **difference in scope** (grid-fed vs. grid-fed + self-consumed) — which confirms and refines the limitation already noted in the conclusion (§12). The risk of an upward revision of the provisional 2025 data, meanwhile, remains real and is documented by the 2022-2024 precedent above.

In [ ]:
raw["nuclear"] = raw["nuclear"].fillna(0)

# National aggregation: sum of the 13 regions, explicit monthly frequency (MS = start of month)
nat = raw.groupby("month")[SOURCES].sum().asfreq("MS")
y = nat["solar"].rename("Solar production (GWh)")

print("Index frequency:", y.index.freqstr, "| missing months:", y.isna().sum())
y.describe().to_frame().T

## 3. Exploratory analysis

### 3.1 The electricity mix: why solar?

In [ ]:
annual = nat.resample("YE").sum() / 1000   # TWh
annual.index = annual.index.year

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
annual.plot(kind="bar", stacked=True, ax=axes[0], colormap="tab20c", width=0.8)
axes[0].set_title("Annual production by source (TWh)")
axes[0].set_xlabel(""); axes[0].legend(fontsize=8, ncol=2)

growth = (annual / annual.loc[2014] * 100)
growth.plot(ax=axes[1], marker="o", ms=3)
axes[1].axhline(100, color="k", lw=0.8, ls="--")
axes[1].set_title("Index, base 100 in 2014"); axes[1].set_xlabel("")
plt.tight_layout(); plt.show()

share = annual.div(annual.sum(axis=1), axis=0) * 100
pd.DataFrame({"2014 (TWh)": annual.loc[2014], "2025 (TWh)": annual.loc[2025],
              "Share 2014 (%)": share.loc[2014], "Share 2025 (%)": share.loc[2025],
              "Multiplier": annual.loc[2025] / annual.loc[2014]})

**Interpretation.**

- *Technical.* **Nuclear** dominates in volume (about 70% of the mix), but its dynamics are mostly driven by shocks: the **2022** dip corresponds to the stress-corrosion-cracking crisis, which took many reactors offline. This is not a regular stochastic process, it is an operational disruption. **Thermal** is structurally declining. **Hydro** tracks rainfall, with no clear trend.
- **Solar** shows the strongest and most regular growth of all sources (×5.5 between 2014 and 2025). It goes from under 1% to about 6% of the mix.
- *Business.* Solar is the source whose **trajectory** matters most for planning. It is also the one whose seasonal swings are becoming a system-level issue (summer surpluses, negative prices). Statistically, it is the series that best exercises the three components we want to model: trend, seasonality and noise.

### 3.2 The national solar series

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y, color="darkorange", lw=1.5, label="Monthly production")
ax.plot(y.rolling(12, center=True).mean(), color="k", lw=2, label="12-month centered moving average (trend)")
ax.plot(y.rolling(12, center=True).std(), color="tab:red", lw=1.5, ls="--", label="12-month rolling std. dev.")
ax.set_title("Monthly solar production — Metropolitan France (GWh)")
ax.set_ylabel("GWh"); ax.legend(); plt.show()

yoy = annual["solar"].pct_change() * 100
print("Year-on-year growth of solar production (%):")
print(yoy.round(1).dropna().to_string())

**Interpretation.**

- *Trend.* The moving average is **convex**. It grows by about 10% per year from 2015 to 2021, then accelerates markedly: **+35% in 2022**, +17% in 2023, +10% in 2024 and **+33% in 2025**. Growth is therefore neither linear nor even at a constant rate.
- *Variance.* The rolling standard deviation increases **proportionally** with the level. The seasonal amplitude (summer/winter gap) grows with the installed fleet. This is the signature of **multiplicative seasonality**: the series is **non-stationary in both mean and variance**.
- *Business.* Production = **installed capacity × capacity factor**. The trend reflects the pace of grid connections (the "how many panels" part). Seasonality reflects sunshine (the "how much sun" part). The multiplicative pattern has a physical explanation: doubling the fleet doubles both June's and December's production, hence the gap between the two.
- **Modeling consequence:** we will need either multiplicative seasonality (ETS), or a **logarithmic transform** that makes seasonality additive (SARIMA, Prophet).

In [ ]:
# Seasonal profile by year (seasonal plot) and normalized profile
tmp = y.to_frame("v"); tmp["year"] = tmp.index.year; tmp["month"] = tmp.index.month
pivot = tmp.pivot(index="month", columns="year", values="v")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
pivot.plot(ax=axes[0], cmap="viridis", legend=False)
axes[0].set_title("Monthly profile by year (GWh)"); axes[0].set_xticks(range(1, 13))
sm = plt.cm.ScalarMappable(cmap="viridis", norm=plt.Normalize(2014, 2025))
fig.colorbar(sm, ax=axes[0], label="Year")

(pivot / pivot.mean()).plot(ax=axes[1], cmap="viridis", legend=False, alpha=0.7)
(pivot / pivot.mean()).mean(axis=1).plot(ax=axes[1], color="red", lw=3, label="Average")
axes[1].set_title("Normalized profile (production / annual average)")
axes[1].set_xticks(range(1, 13)); axes[1].legend()
plt.tight_layout(); plt.show()

**Interpretation.**

- *Technical.* On the left, the curves stack and **spread out** year after year, confirming the multiplicative pattern. On the right, once each year is divided by its own average, the profiles **nearly overlap perfectly**. The *shape* of seasonality is therefore stable over time: a peak from June to August, a trough from November to January.
- *Business.* Solar seasonality is an **astronomical** phenomenon (day length, sun elevation): there is no reason for it to change from one year to the next. The residual differences between years come from the weather (an unusually sunny spring 2022, for instance). A good model should therefore estimate a **near-deterministic** seasonality, and we will check this hypothesis in each model.

In [ ]:
# Regional breakdown
reg = raw.pivot_table(index=raw["month"].dt.year, columns="region", values="solar", aggfunc="sum")
part = (reg.loc[2025] / reg.loc[2025].sum() * 100).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 4))
part.plot(kind="bar", ax=ax, color="darkorange")
ax.set_title("Share of each region in national solar production in 2025 (%)")
ax.set_xlabel(""); plt.xticks(rotation=45, ha="right"); plt.show()
print(f"Top 4 regions: {part.head(4).sum():.0f}% of production")

**Interpretation.** Production is concentrated in the **south and west**: Nouvelle-Aquitaine, Occitanie, Auvergne-Rhône-Alpes and PACA together supply about **64%** of the total. Nouvelle-Aquitaine has overtaken Occitanie, which was the leading region in 2014, thanks to its large ground-mounted plants.

*Business.* The national forecast therefore hides a **geographic** issue. Summer surpluses are concentrated in the south-west, which puts pressure on transmission lines toward consumption areas. A natural extension of this project would be to forecast region by region (see Limitations).

## 4. Decomposition of the series

We split the series into **trend (T)**, **seasonality (S)** and **residual (R)**:
- additive: $y_t = T_t + S_t + R_t$
- multiplicative: $y_t = T_t \times S_t \times R_t \iff \log y_t = \log T_t + \log S_t + \log R_t$

### 4.1 Classical decomposition: additive vs. multiplicative

In [ ]:
dec_add = seasonal_decompose(y, model="additive", period=12)
dec_mul = seasonal_decompose(y, model="multiplicative", period=12)

fig, axes = plt.subplots(4, 2, figsize=(15, 10), sharex=True)
for j, (d, name) in enumerate([(dec_add, "Additive"), (dec_mul, "Multiplicative")]):
    for i, (comp, lab) in enumerate([(d.observed, "Observed"), (d.trend, "Trend"),
                                     (d.seasonal, "Seasonal"), (d.resid, "Residual")]):
        axes[i, j].plot(comp, color="tab:blue" if i < 3 else "tab:red",
                        marker="." if i == 3 else None, ls="-" if i < 3 else "")
        axes[i, j].set_ylabel(lab)
        if i == 3:
            axes[i, j].axhline(0 if name == "Additive" else 1, color="k", lw=0.8)
    axes[0, j].set_title(f"{name} decomposition")
plt.tight_layout(); plt.show()

**Interpretation.**

- In the **additive** decomposition, the seasonal component is constrained to a fixed amplitude (about ±1,000 GWh). The residuals then inherit whatever seasonality fails to capture: they form a **funnel** shape, small at first and large at the end. This is a sign of misspecification, since structure remains in the residuals.
- In the **multiplicative** decomposition, the residuals oscillate around 1 with a **roughly constant** dispersion over time. This is the correct model.
- *Decision:* we keep the **multiplicative structure** for the rest of the analysis.

In [ ]:
idx = pd.Series(dec_mul.seasonal.iloc[:12].values,
                index=["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(idx.index, (idx - 1) * 100, color=np.where(idx > 1, "darkorange", "steelblue"))
ax.bar_label(bars, fmt="%+.0f%%"); ax.axhline(0, color="k", lw=0.8)
ax.set_title("Multiplicative seasonal coefficients: deviation from trend (%)"); plt.show()
print(f"July / December ratio: {idx['Jul'] / idx['Dec']:.1f}")

**Interpretation of the seasonal coefficients.**

- *Technical.* A coefficient of 1.53 in July means that July production is on average **53% above trend**. December is at **−62%**. The July/December ratio is about **4**. The coefficients are normalized so that their yearly average equals 1.
- *Business:*
  - **April to August** (+22% to +53%) is the period when solar weighs most heavily on the system. It is also the period of lowest demand (no heating needed), hence the episodes of **negative prices** and **curtailment**. It is the preferred window for scheduling nuclear maintenance outages.
  - **November to January** (−48% to −62%) corresponds to the winter consumption peak, and solar contributes very little to it. It cannot be relied on for winter supply security, which remains ensured by nuclear, hydro and imports.
  - For a producer, about **58% of annual revenue** (by volume) is generated between April and August, which has a direct impact on cash flow and banking covenants.

### 4.2 Robust STL decomposition on the logarithm

Classical decomposition has two limitations: it loses 6 points at each end (centered moving average, or **CMA** — the method that is precisely used to isolate $T_t$ by smoothing out $S_t$ and $R_t$) and it imposes a seasonality that is strictly identical every year.

**Methodological note — X-11 / X-13ARIMA-SEATS.** These are the "official" seasonal-adjustment procedures (Census Bureau, used by Insee and Eurostat to publish seasonally-adjusted series). X-11 refines the CMA principle through iterative moving averages with extreme-value treatment; X-13ARIMA-SEATS adds an ARIMA extrapolation at the ends of the series (so no points are lost) and an option for decomposition based on an ARIMA model (SEATS) rather than filters. They are the standard in official statistics, but rely on filters designed for series that are already long and stable.

We do not apply them here and instead use **STL** (*Seasonal-Trend decomposition using LOESS*, Cleveland et al. 1990), for three reasons suited to our case: (i) it loses no points at the ends, which matters for a short series (144 months); (ii) its `robust` mode natively withstands atypical months (weather anomalies, see 4.3), with no extra tuning unlike X-11; (iii) it allows seasonality to evolve freely over time rather than following a rigid pattern, which lets us **verify** — rather than assume — that solar seasonality is stable. We apply it to $\log y_t$, which amounts to a multiplicative decomposition.

In [ ]:
ly = np.log(y)
stl = STL(ly, period=12, robust=True).fit()
fig = stl.plot(); fig.set_size_inches(14, 8); plt.tight_layout(); plt.show()

# Strength of trend and seasonality (Wang, Smith & Hyndman, 2006)
T, S, R = stl.trend, stl.seasonal, stl.resid
F_T = max(0, 1 - R.var() / (T + R).var())
F_S = max(0, 1 - R.var() / (S + R).var())
print(f"Trend strength      F_T = {F_T:.3f}")
print(f"Seasonal strength    F_S = {F_S:.3f}")

**Interpretation.**

- *Technical.* Both strength indicators ($F_T \approx 0.97$ and $F_S \approx 0.96$, on a scale from 0 to 1) are **very high**. More than 95% of the variability is explained by trend and seasonality, and the irregular noise is small. This is a **structurally very predictable** series.
- The log-trend is nearly a straight line up to around 2020, after which its slope increases from 2021–2022 onward. A straight line in log terms corresponds to **exponential growth** at a constant rate. The acceleration means that the growth rate itself has increased.
- *Business.* The main forecasting risk is **not** the weather (the noise is small), but the **pace of fleet deployment**. If grid connections accelerate or slow down (CRE tender rounds, panel costs, changes in feed-in tariffs), the trend shifts and models that extrapolate the past will be wrong. We will see this clearly in 2025.

In [ ]:
# The most extreme STL residuals: which months were atypical?
resid_pct = (np.exp(R) - 1) * 100
top = resid_pct.abs().sort_values(ascending=False).head(8).index
pd.DataFrame({"STL residual (% vs. expected)": resid_pct[top].round(1),
              "Production (GWh)": y[top].round(0)}).sort_index()

**Interpretation of the anomalies.**

- *Technical.* The eight largest relative residuals (from −39% to +40%) **all fall between November and February**. This is no coincidence: in winter production is low (200 to 800 GWh), so a few extra sunny or overcast days are enough to shift a month by 25 to 40% in relative terms. In GWh, these gaps remain modest. These are **weather effects**: December 2014 was particularly overcast, January 2022 particularly sunny, and so on.
- *For the team to do:* cross-reference these months with Météo-France's monthly climate reports to confirm and document this.
- *Business.* **Weather risk** is strong in relative terms in winter, but it weighs most heavily in volume (and therefore in euros) in summer. Methodological consequence: **MAPE** will be dominated by winter errors, while **RMSE** will be dominated by summer ones (see section 9).

## 5. Evaluation protocol and baseline

### 5.1 Time split
- **Training:** January 2014 → December 2024 (132 months)
- **Test:** January → December 2025 (12 months), a realistic annual planning horizon

The split is **chronological**. A random split would leak information from the future into the past: the model would already "know" 2025 while estimating the trend.

Models are **selected on the training set only** (AIC criterion), never on the test set. Choosing a model because it gives the best test error would amount to a form of overfitting to the test set.

### 5.2 Metrics
| Metric | Formula | Reading | When to use it |
|---|---|---|---|
| **MSE** | $\frac{1}{n}\sum e_t^2$ | GWh² (not very readable) | Reference loss function; heavily penalizes large errors |
| **RMSE** | $\sqrt{\text{MSE}}$ | GWh | When one large error costs much more than several small ones (grid security) |
| **MAE** | $\frac{1}{n}\sum \lvert e_t \rvert$ | GWh | "Typical" error, robust to atypical months |
| **MAPE** | $\frac{100}{n}\sum \lvert e_t / y_t \rvert$ | % | Communication to managers; caution, it overweights winter months (small $y_t$) |
| **MASE** | $\text{MAE} / \text{MAE}_{\text{seasonal naive, train}}$ | unitless | Scale-independent comparison; < 1 means better than the naive method |
| **Bias (ME)** | $\frac{1}{n}\sum e_t$ | GWh | Systematic error: under- or over-forecasting |
| **95% CI coverage** | % of actual points inside the interval | % | Reliability of the stated uncertainty |

Here $e_t = y_t - \hat y_t$: a **positive error means under-forecasting**.

In [ ]:
train, test = y[:"2024"], y["2025"]
H = len(test)
print(f"Train: {train.index[0]:%Y-%m} → {train.index[-1]:%Y-%m} ({len(train)} months)")
print(f"Test : {test.index[0]:%Y-%m} → {test.index[-1]:%Y-%m} ({len(test)} months)")

def evaluate(y_true, y_pred, y_train, lower=None, upper=None, m=12):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    e = y_true - y_pred
    scale = np.mean(np.abs(np.asarray(y_train)[m:] - np.asarray(y_train)[:-m]))
    out = {"MSE": np.mean(e**2), "RMSE": np.sqrt(np.mean(e**2)), "MAE": np.mean(np.abs(e)),
           "MAPE (%)": np.mean(np.abs(e / y_true)) * 100, "MASE": np.mean(np.abs(e)) / scale,
           "Bias (GWh)": np.mean(e)}
    if lower is not None:
        out["95% CI coverage (%)"] = np.mean((y_true >= np.asarray(lower)) & (y_true <= np.asarray(upper))) * 100
    return out

forecasts, intervals, results = {}, {}, {}

In [ ]:
# Baseline: seasonal naive  ŷ(t) = y(t-12)
naive = pd.Series(train[-12:].values, index=test.index)
forecasts["Seasonal naive"] = naive
results["Seasonal naive"] = evaluate(test, naive, train)
pd.DataFrame(results).T

**Interpretation.** The seasonal naive method forecasts that 2025 will be identical to 2024. Its bias is strongly positive: it **systematically under-forecasts**, because it ignores fleet growth. This is the minimal benchmark. A model that does not clearly beat it adds no value.

*Business.* This is the "spreadsheet" method many operational teams effectively use ("let's just take last year's numbers"). The gain from statistical models is measured against it.

## 6. ETS Model (Error, Trend, Seasonal)

Exponential smoothing updates three states (level $\ell_t$, slope $b_t$, season $s_t$) with weights that decay exponentially into the past. The state-space **ETS** formulation (Hyndman et al.) adds an explicit error term. This yields a likelihood, hence an AIC, and forecast intervals.

For the multiplicative Holt-Winters model (A, A, M):
$$\hat y_{t+h|t} = (\ell_t + h\,b_t)\, s_{t+h-12}$$
$$\ell_t = \alpha \frac{y_t}{s_{t-12}} + (1-\alpha)(\ell_{t-1}+b_{t-1}), \quad b_t = \beta^*(\ell_t-\ell_{t-1}) + (1-\beta^*)b_{t-1}, \quad s_t = \gamma\frac{y_t}{\ell_{t-1}+b_{t-1}} + (1-\gamma)s_{t-12}$$

### 6.1 Model selection by AIC

In [ ]:
rows = []
for err, trend, damped, seas in itertools.product(["add", "mul"], ["add", None], [False, True], ["add", "mul"]):
    if trend is None and damped:
        continue
    try:
        m = ETSModel(train, error=err, trend=trend, damped_trend=damped,
                     seasonal=seas, seasonal_periods=12).fit(disp=False)
        label = f"ETS({err[0].upper()},{'N' if trend is None else ('Ad' if damped else 'A')},{seas[0].upper()})"
        rows.append({"Model": label, "error": err, "trend": trend, "damped": damped,
                     "seasonal": seas, "AIC": m.aic, "AICc": m.aicc, "BIC": m.bic})
    except Exception as ex:
        print("failed", err, trend, damped, seas, ex)

ets_grid = pd.DataFrame(rows).sort_values("AICc").reset_index(drop=True)
ets_grid[["Model", "AIC", "AICc", "BIC"]]

**Interpretation.**

- The six best models **all have multiplicative seasonality (M)**, with a gap of about 100 AIC points relative to the additive versions. This is a strong statistical confirmation of what the EDA and the decomposition suggested.
- The best model is **ETS(A,A,M)**: additive error, additive trend, multiplicative seasonality. This is the classic multiplicative Holt-Winters model.
- The **damped-trend** version (Ad) ranks lower. The data show no sign of growth slowing down, and damping would impose a deceleration that the past does not justify.

In [ ]:
best = ets_grid.iloc[0]
ets_fit = ETSModel(train, error=best["error"], trend=best["trend"], damped_trend=best["damped"],
                   seasonal=best["seasonal"], seasonal_periods=12).fit(disp=False)
print(best["Model"])
p = pd.Series(np.asarray(ets_fit.params), index=ets_fit.param_names)
pd.DataFrame({"Value": [p["smoothing_level"], p["smoothing_trend"], p["smoothing_seasonal"]]},
             index=["α (level)", "β (slope)", "γ (seasonal)"])

**Interpretation of the smoothing parameters** (each between 0 and 1; the larger it is, the more the state reacts to recent observations):

| Parameter | Value | Technical reading | Business reading |
|---|---|---|---|
| **α ≈ 0.24** | moderate | The level incorporates about a quarter of each new observation. Information "dilutes" over roughly half a dozen months. | The fleet's level adjusts gradually. An unusually sunny month does not make the model believe the fleet has suddenly jumped. |
| **β ≈ 0.01** | very low | The slope (monthly growth) is nearly fixed and is revised only very slowly. | The model assumes a **stable connection pace**. This is its weakness in 2022 and 2025, when that pace accelerated. |
| **γ ≈ 0** | near zero | The seasonal coefficients barely move: **deterministic** seasonality. | This result is consistent with the physics: the annual solar cycle does not change. The model "discovers" on its own what we already saw in section 3.2. |

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
axes[0].plot(ets_fit.level, color="tab:blue"); axes[0].set_ylabel("Level ℓ (GWh)")
axes[0].set_title(f"States estimated by {best['Model']}")
axes[1].plot(ets_fit.slope, color="tab:green"); axes[1].set_ylabel("Slope b (GWh/month)")
axes[2].plot(ets_fit.season, color="darkorange"); axes[2].set_ylabel("Season s")
plt.tight_layout(); plt.show()
print(f"Final slope: {ets_fit.slope.iloc[-1]:.1f} GWh/month, i.e. ≈ {ets_fit.slope.iloc[-1]*12:.0f} GWh of "
      f"extra 'deseasonalized' monthly production added per year")

**Interpretation of the states.**

- The **slope** goes from about 2 to about 10 GWh/month over the period. Each month, the deseasonalized level gains this volume, which corresponds to newly connected installations.
- The **season** state is nearly flat, confirming γ ≈ 0.
- *Business.* The slope is a leading indicator of the connection pace. If it slowed down while the PPE targets an acceleration, this would be a warning signal for public authorities.

In [ ]:
pred = ets_fit.get_prediction(start=test.index[0], end=test.index[-1]).summary_frame(alpha=0.05)
forecasts["ETS"] = pred["mean"]; intervals["ETS"] = (pred["pi_lower"], pred["pi_upper"])
results["ETS"] = evaluate(test, pred["mean"], train, pred["pi_lower"], pred["pi_upper"])

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(y["2021":], color="k", label="Actual")
ax.plot(pred["mean"], color=COLORS["ETS"], lw=2, label="ETS forecast")
ax.fill_between(pred.index, pred["pi_lower"], pred["pi_upper"], color=COLORS["ETS"], alpha=0.2, label="95% CI")
ax.axvline(test.index[0], color="grey", ls="--"); ax.legend(); ax.set_title("ETS: 2025 forecast vs. actual")
plt.show()

# Residual diagnostics
res_ets = ets_fit.resid
lb = acorr_ljungbox(res_ets, lags=[12, 24], return_df=True)
print("Ljung-Box (H0: residuals are not autocorrelated)\n", lb.round(4))
print(f"Shapiro-Wilk p-value (normality): {stats.shapiro(res_ets)[1]:.4f}")
pd.DataFrame(results).T

**Interpretation.**

- *Forecast.* ETS reproduces the **shape** of 2025 well (summer peak, winter trough), but it **underestimates the whole summer** (an error of about 700 GWh in July). The average bias is positive: the model extrapolated the 2014–2024 growth pace, whereas 2025 saw growth of +33%, the strongest in the historical record.
- *Residuals.* The Ljung-Box test rejects the absence of autocorrelation at lag 12 (p ≈ 0.01): some information remains unexploited. The Shapiro-Wilk test also rejects normality (p ≈ 0.03).
- *Intervals.* Only **5 out of 12 months (42%)** fall inside the 95% interval. This is far from the stated 95%: the model is **far too confident**. The cause is structural. An **additive** error assumes a constant variance in GWh, estimated over 2014–2024, when production was 3 to 5 times smaller. Applied to summer-2025 volumes, this variance is severely underestimated.
- *Business.* The model does much better than "last year," but a planner who had used it would have **undersized** the summer-2025 flexibility needs, and would have wrongly believed they were well covered by the confidence interval.

### 6.4 Robustness check: ETS(M,A,M) multiplicative error
The previous diagnostic suggests trying a **multiplicative** error, whose variance is proportional to the level. This is the second-ranked model by AIC.

In [ ]:
ets_mam = ETSModel(train, error="mul", trend="add", seasonal="mul", seasonal_periods=12).fit(disp=False)
p_mam = ets_mam.get_prediction(start=test.index[0], end=test.index[-1]).summary_frame(alpha=0.05)
comp = pd.DataFrame({"ETS(A,A,M)": evaluate(test, pred["mean"], train, pred["pi_lower"], pred["pi_upper"]),
                     "ETS(M,A,M)": evaluate(test, p_mam["mean"], train, p_mam["pi_lower"], p_mam["pi_upper"])}).T
comp["Avg. 95% CI width (GWh)"] = [np.mean(pred["pi_upper"] - pred["pi_lower"]), np.mean(p_mam["pi_upper"] - p_mam["pi_lower"])]
comp["Ljung-Box p (lag 12)"] = [acorr_ljungbox(ets_fit.resid, lags=[12])["lb_pvalue"].iloc[0],
                                acorr_ljungbox(ets_mam.resid, lags=[12])["lb_pvalue"].iloc[0]]
comp

**Interpretation: a precision / uncertainty-reliability trade-off.**

- ETS(M,A,M) produces **residuals with no autocorrelation** (Ljung-Box not significant) and intervals twice as wide, which cover about **83%** of the months of 2025, much closer to the promised 95%.
- On the other hand, its **point forecast is worse** (MAE ≈ 450 GWh vs. ≈ 315).
- *Methodological lesson:* AIC measures the fit to past data, not forecast quality or interval quality. The two must be evaluated separately.
- *Business:* if the main use is **risk sizing** (grid safety margins, volume hedging), ETS(M,A,M) is preferable. If it is a **central estimate** (budgeting), ETS(A,A,M) is. For the rest of the notebook, we keep ETS(A,A,M), selected by AIC, as the representative of the ETS family.

## 7. SARIMA Model

A SARIMA$(p,d,q)(P,D,Q)_{12}$ models the series (made stationary through differencing) as a combination of its own past values (AR), its past shocks (MA), and the same terms 12 months apart (seasonal).

ARIMA assumes **constant variance**. We therefore work on $\log y_t$, which makes seasonality additive and stabilizes the variance (see section 4).

### 7.1 Stationarity
We use two tests with opposite null hypotheses, to guard against false positives:
- **ADF**: H₀ = unit root (non-stationary). A p-value < 0.05 indicates a stationary series.
- **KPSS**: H₀ = stationary. A p-value < 0.05 indicates a non-stationary series.

In [ ]:
lt = np.log(train)
def stationarity(s, name):
    s = s.dropna()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return {"Series": name, "ADF p-value": adfuller(s, autolag="AIC")[1],
                "KPSS p-value": kpss(s, regression="c", nlags="auto")[1]}

pd.DataFrame([
    stationarity(lt, "log(y)"),
    stationarity(lt.diff(), "Δ log(y)  (d=1)"),
    stationarity(lt.diff(12), "Δ₁₂ log(y)  (D=1)"),
    stationarity(lt.diff(12).diff(), "Δ Δ₁₂ log(y)  (d=1, D=1)"),
]).set_index("Series").round(4)

**Interpretation.** (The KPSS p-value is truncated between 0.01 and 0.10 by the test's tables.)

- **log(y)**: the ADF test does not reject the presence of a **unit root** (H₀ of the ADF test) and the KPSS test rejects stationarity. The series is **non-stationary**, as expected given the trend.
- **Δ log(y)** (simple difference): both tests conclude in favor of stationarity, but ADF and KPSS only test for a **non-seasonal** unit root. Seasonality is still there (an ACF spike at lag 12). A seasonal difference is therefore also needed.
- **Δ₁₂ log(y)** (seasonal difference): both tests agree on **stationarity**. This series has a direct business meaning: $\log y_t - \log y_{t-12} \approx$ **year-on-year growth rate**. It is the annual growth of solar production, month by month.
- The double difference is also stationary, but it risks **over-differencing**, which introduces an artificial MA structure.
- *Decision:* **D = 1** is necessary. For **d ∈ {0, 1}**, we let the AIC decide. With d = 0, a constant in the model represents an average annual growth rate.

**Stochastic vs. deterministic trend.** The unit root not rejected by the ADF test on log(y) is the signature of a **stochastic trend**: $\log y_t = \log y_{t-1} + c + \varepsilon_t$ (random walk with drift). A shock (a particularly strong or weak year of deployment) then has a **permanent** effect — the series never returns to its old trajectory, it adopts a new one. This is consistent with what we observe in 2022 and 2025: an acceleration in the connection pace durably shifts the level of the series; it does not "catch up" to its old slope. The appropriate treatment is **differencing** (what we do here), not the removal of a deterministic trend of the form $\log y_t = \alpha + \beta t + \varepsilon_t$ (regression on time). Confusing the two would be a mistake: a deterministic trend is removed by linear regression on $t$, a stochastic trend is removed by differencing — applying the wrong method either leaves non-stationary structure in the residuals, or over-differences a series that did not need it.

In [ ]:
d12 = lt.diff(12).dropna()
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(np.exp(d12) * 100 - 100, color="tab:purple")
axes[0].axhline(0, color="k", lw=0.8); axes[0].set_title("Rolling 12-month growth (%)")
plot_acf(d12, lags=36, ax=axes[1], title="ACF of Δ₁₂ log(y)")
plot_pacf(d12, lags=36, ax=axes[2], title="PACF of Δ₁₂ log(y)", method="ywm")
plt.tight_layout(); plt.show()

**Interpretation.**

- *Left panel (business).* Annual growth oscillates between about 0 and +60%. It is strong and volatile early on (small base), falls to its lowest point in 2020–2021 (Covid crisis, construction delays), then rises again from 2022.
- *ACF.* The autocorrelation decays **slowly** over the first few lags, a behavior typical of an **AR** component close to the unit root: one month's growth resembles the previous month's. A negative spike at lag 12 suggests a **seasonal MA term (Q = 1)**, the classic signature of a seasonal difference.
- *PACF.* A sharp cutoff after lag 1 or 2 points to an **AR(1) or AR(2)**.
- *Candidates:* p ∈ {1, 2}, q ∈ {0, 1, 2}, Q = 1. We confirm with a systematic grid search.

In [ ]:
import statsmodels.tools.sm_exceptions as smx
warnings.simplefilter("ignore", smx.ConvergenceWarning); warnings.simplefilter("ignore", smx.EstimationWarning)
rows = []
for p_, d_, q_, P_, Q_ in itertools.product([0, 1, 2], [0, 1], [0, 1, 2], [0, 1], [0, 1]):
    try:
        m = SARIMAX(lt, order=(p_, d_, q_), seasonal_order=(P_, 1, Q_, 12),
                    trend="c" if d_ == 0 else "n").fit(disp=False)
        rows.append({"Model": f"({p_},{d_},{q_})({P_},1,{Q_})12", "order": (p_, d_, q_),
                     "sorder": (P_, 1, Q_, 12), "AIC": m.aic, "BIC": m.bic})
    except Exception:
        pass
sar_grid = pd.DataFrame(rows).sort_values("AIC").reset_index(drop=True)
print(f"{len(sar_grid)} models estimated")
sar_grid[["Model", "AIC", "BIC"]].head(10)

**Interpretation.** The best models are very close in AIC (less than 2 points apart, hence statistically indistinguishable). They all share the same structure: **a strong AR(1) or AR(2), some MA, and Q = 1**. We keep the top-ranked one by AIC.

Note: SARIMA's AIC (on the log) **is not comparable** to ETS's (on raw GWh). The likelihoods are not computed on the same variable. Comparison between families is done solely on forecast errors (section 9).

In [ ]:
sel = sar_grid.iloc[0]
sar_fit = SARIMAX(lt, order=sel["order"], seasonal_order=sel["sorder"],
                  trend="c" if sel["order"][1] == 0 else "n").fit(disp=False)
print("SARIMA", sel["Model"], "on log(y)")
sar_fit.summary().tables[1]

**Interpretation of the coefficients** (on the seasonally-differenced $\log y$ series, i.e. 12-month growth):

- **Constant ≈ 0.01**: since d = 0 and D = 1, it relates to the long-run average annual growth. It is not significant on its own, because the strong AR persistence already "carries" the level.
- **AR(1) ≈ 0.93**: very close to 1. 12-month growth is **highly persistent**. If the fleet grows faster than usual this month, it will probably keep doing so next month. *Business:* the deployment pace has strong inertia (construction projects last several months, tenders run over years).
- **MA(1) ≈ −0.60**: part of each shock is **corrected** the following month. *Business:* this is the **weather** component of the shock. A very sunny month inflates 12-month growth, but the effect does not persist.
- **Seasonal MA (Θ ≈ −1)**: a value at the boundary of invertibility. Θ = −1 exactly "cancels out" the seasonal difference. This means seasonality is **deterministic**, which is **exactly the same conclusion** as γ ≈ 0 in ETS. Two different methods converge on the same physical diagnosis. The huge standard error on this coefficient is a consequence of sitting at that boundary, not a problem with the model.

In [ ]:
fig = sar_fit.plot_diagnostics(figsize=(14, 8)); plt.tight_layout(); plt.show()
lb = acorr_ljungbox(sar_fit.resid[13:], lags=[12, 24], return_df=True)
print("Ljung-Box on SARIMA residuals\n", lb.round(4))

**Interpretation of the diagnostics.**

- *Standardized residuals:* no visible trend or seasonality remains. The variance is fairly stable, showing that the log transform has done its job.
- *Histogram / QQ-plot:* the distribution is close to normal, with a few atypical months in the tails (the weather anomalies from section 4).
- *Correlogram and Ljung-Box:* a p-value > 0.05 indicates that there is **no more significant autocorrelation**. In other words, the residuals behave like **white noise** (zero mean, constant variance, zero autocovariance at every non-zero lag) — the statistical signature we look for in the residuals of a well-specified model. The model has extracted all the available linear information. Forecast intervals are then reliable, provided the future resembles the past.

In [ ]:
fc = sar_fit.get_forecast(H)
ci = fc.conf_int(alpha=0.05)
sar_mean = np.exp(fc.predicted_mean)           # median of y (exp of the log mean)
sar_lo, sar_hi = np.exp(ci.iloc[:, 0]), np.exp(ci.iloc[:, 1])
forecasts["SARIMA"] = sar_mean; intervals["SARIMA"] = (sar_lo, sar_hi)
results["SARIMA"] = evaluate(test, sar_mean, train, sar_lo, sar_hi)

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(y["2021":], color="k", label="Actual")
ax.plot(sar_mean, color=COLORS["SARIMA"], lw=2, label="SARIMA forecast")
ax.fill_between(test.index, sar_lo, sar_hi, color=COLORS["SARIMA"], alpha=0.2, label="95% CI")
ax.axvline(test.index[0], color="grey", ls="--"); ax.legend(); ax.set_title("SARIMA: 2025 forecast vs. actual")
plt.show()
pd.DataFrame(results).T

**Interpretation.**

- *Technical.* Back-transforming with $\exp$ gives the **median** of the forecast, not its mean. The bias correction $\exp(\sigma^2/2)$ would be about +0.5% here, which is negligible. The intervals are **asymmetric**, wider toward the top, which is consistent with a positive variable under multiplicative growth.
- *Forecast.* SARIMA also shows an **under-forecasting** bias, especially in summer (−885 GWh in July). It is slightly less accurate than ETS on 2025. However, its 95% intervals contain **all 12** realized months: unlike ETS(A,A,M), its uncertainty is honest, which we owe to working in logs.
- *Business.* The 95% intervals widen with the horizon. In December 2025 (12-month horizon), the uncertainty is much larger than in January. For an annual budget, this argues for **revising the forecast every month** rather than once a year.

## 8. Prophet Model

Prophet (Meta, Taylor & Letham 2018) is a **generalized additive model**:
$$y_t = g(t) + s(t) + h(t) + \varepsilon_t$$
- $g(t)$: a **piecewise-linear** trend, with *changepoints* where the slope can change;
- $s(t)$: seasonality as a **Fourier series**;
- $h(t)$: holiday effects (not applicable to monthly data).

Unlike ETS and SARIMA, Prophet does **not** model the autocorrelation of errors. It is a *curve-fitting* approach rather than a stochastic process.

**Specification choice:** we fit Prophet on $\log y$ in additive mode. A piecewise-linear trend in log corresponds to a **piecewise-exponential** growth in GWh, which is more realistic for a fleet under deployment. The "multiplicative on raw data" variant is tested in 8.3.

In [ ]:
def fit_prophet(series, log=True, mode="additive", interval=0.95, cps=0.05):
    df_p = pd.DataFrame({"ds": series.index, "y": np.log(series.values) if log else series.values})
    m = Prophet(seasonality_mode=mode, yearly_seasonality=True, weekly_seasonality=False,
                daily_seasonality=False, interval_width=interval, changepoint_prior_scale=cps)
    m.fit(df_p)
    return m

def predict_prophet(m, periods, log=True):
    fut = m.make_future_dataframe(periods=periods, freq="MS")
    f = m.predict(fut)
    if log:
        for c in ["yhat", "yhat_lower", "yhat_upper"]:
            f[c] = np.exp(f[c])
    return f.set_index("ds")

pr_fit = fit_prophet(train)
pr_full = predict_prophet(pr_fit, H)
pr_fc = pr_full.loc[test.index]
forecasts["Prophet"] = pr_fc["yhat"]; intervals["Prophet"] = (pr_fc["yhat_lower"], pr_fc["yhat_upper"])
results["Prophet"] = evaluate(test, pr_fc["yhat"], train, pr_fc["yhat_lower"], pr_fc["yhat_upper"])

In [ ]:
from prophet.plot import add_changepoints_to_plot
fig = pr_fit.plot(pr_fit.predict(pr_fit.make_future_dataframe(H, freq="MS")), figsize=(14, 5))
add_changepoints_to_plot(fig.gca(), pr_fit, pr_fit.predict(pr_fit.make_future_dataframe(H, freq="MS")))
plt.title("Prophet (log scale): fit, forecast and changepoints"); plt.show()

fig = pr_fit.plot_components(pr_fit.predict(pr_fit.make_future_dataframe(H, freq="MS")), figsize=(14, 6))
plt.show()

In [ ]:
# Implied annual growth rate of the trend, segment by segment between changepoints
trend_log = pr_full["trend"] if "trend" in pr_full else None
tr = pr_fit.predict(pr_fit.make_future_dataframe(0, freq="MS")).set_index("ds")["trend"]
g = (np.exp(tr.diff(12)) - 1) * 100
g_year = g.groupby(g.index.year).mean().dropna()
print("Implied annual growth rate of the Prophet trend (%):")
print(g_year.round(1).to_string())

deltas = pd.Series(pr_fit.params["delta"].mean(axis=0), index=pr_fit.changepoints.dt.strftime("%Y-%m"))
print("\nChangepoints with the largest slope change:")
print(deltas.abs().sort_values(ascending=False).head(5).round(4).to_string())

**Interpretation.**

- *Trend and changepoints.* Prophet identifies **slope breaks** in the log trend. The implied trend growth goes from about 20% per year in 2015 to about **11% per year from 2017 to 2020**. The largest slope changes are dated **August 2020 to April 2021**, after which growth rises back to **18–21% per year** (2021–2024). This is consistent with the sector's history: a plateau in the late 2010s, then a post-Covid rebound amplified by the 2022 surge in electricity prices. *(To be substantiated with sources: CRE tender rounds, SDES dashboard.)*
- The 2024 trend rate (≈ 18%) is well above the growth actually observed that year (+10%). Prophet smooths across years and does not follow every bump, which led it to extrapolate stronger growth for 2025 than the other models.
- *Seasonality (yearly component).* Since we work in logs, the component is an additive deviation in log terms, hence a **percentage** relative to the trend. Its profile is identical to that of the coefficients in section 4.1, showing that the three approaches converge.
- *Strengths of Prophet:* the changepoints are **interpretable** and dated, which is very useful for a business report ("growth shifted regime in 2022").
- *Limitation:* beyond the last changepoint, Prophet **extrapolates the last slope** linearly in log terms. It cannot anticipate a new acceleration. The `changepoint_prior_scale` parameter controls flexibility: a higher value makes the trend more reactive but increases the risk of overfitting.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(y["2021":], color="k", label="Actual")
ax.plot(pr_fc["yhat"], color=COLORS["Prophet"], lw=2, label="Prophet forecast")
ax.fill_between(test.index, pr_fc["yhat_lower"], pr_fc["yhat_upper"], color=COLORS["Prophet"], alpha=0.2, label="95% CI")
ax.axvline(test.index[0], color="grey", ls="--"); ax.legend(); ax.set_title("Prophet: 2025 forecast vs. actual")
plt.show()
pd.DataFrame(results).T

### 8.3 Variant: multiplicative Prophet on raw data

In [ ]:
pr_mul = fit_prophet(train, log=False, mode="multiplicative")
f_mul = predict_prophet(pr_mul, H, log=False).loc[test.index]
pd.DataFrame({"Prophet log-additive": evaluate(test, pr_fc["yhat"], train),
              "Prophet raw multiplicative": evaluate(test, f_mul["yhat"], train)}).T

**Interpretation.** The multiplicative version on raw data is less accurate. Its trend $g(t)$ stays **linear in GWh**, which amounts to assuming the same volume of panels is added every year. The log version assumes that each year adds a **percentage** of the existing fleet, which matches the observed dynamics better. This is a good example of how a specification choice grounded in understanding of the phenomenon improves the forecast.

## 9. Methodological Comparison of Forecasts

### 9.1 Performance on the 2025 test set

In [ ]:
res_df = pd.DataFrame(results).T
res_df.style.format("{:,.2f}").highlight_min(subset=["MSE", "RMSE", "MAE", "MAPE (%)", "MASE"], color="#c6efce")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
axes[0].plot(y["2023":], color="k", lw=2.5, label="Actual")
for name, f in forecasts.items():
    axes[0].plot(f, color=COLORS[name], lw=1.8, ls="--" if name == "Seasonal naive" else "-", label=name)
axes[0].axvline(test.index[0], color="grey", ls=":"); axes[0].legend(); axes[0].set_title("2025 forecasts")

err = pd.DataFrame({n: test - f for n, f in forecasts.items()})
err.index = err.index.strftime("%b")
err.plot(kind="bar", ax=axes[1], color=[COLORS[c] for c in err.columns], width=0.8)
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_title("Monthly error = actual − forecast (GWh); > 0 = under-forecast")
plt.tight_layout(); plt.show()

**Interpretation.**

- **All three models comfortably beat the seasonal naive baseline**: MAPE drops from 24% to 8–13%. Explicitly modeling the trend has real value.
- **Prophet is the most accurate on every metric in 2025** (MAE ≈ 245 GWh, MAPE ≈ 8.5%), ahead of ETS (≈ 314 GWh) and SARIMA (≈ 362 GWh). The explanation lies in how each model extrapolates the trend:
  - **ETS(A,A,M)** adds a **fixed number of GWh** to the level each month. In percentage terms, the growth it projects therefore **mechanically decreases** as the fleet grows.
  - **SARIMA** on the log models a growth rate, but its AR(1) < 1 pulls it back **toward the historical average growth** (weaker than recent growth).
  - **Prophet** on the log extends the **recent slope** (≈ 18% per year): this turned out to be the assumption closest to the reality of 2025.
- **All models under-forecast 2025**: all 36 bars (3 models × 12 months) are positive, with the largest from May to August. The error is therefore not random: it is a **systematic bias** shared by every method. 2025 saw growth (+33%) higher than anything the historical record would have suggested. No purely univariate model, which extrapolates the past, could have anticipated it. That information lies in **exogenous variables**: connected capacity, connection-queue backlog, sunshine.
- **Why the error peaks in summer:** with a multiplicative structure, an x% error on the trend turns into a GWh error proportional to the seasonal level. It is therefore roughly 4 times larger in July than in December.
- **Ranking by metric:** for 2025, the ranking is the same regardless of the metric used. This is not always the case: in cross-validation (9.3), Prophet and SARIMA tie on MAPE but diverge on RMSE. MSE/RMSE penalizes large summer errors, MAPE weighs winter months more heavily. The choice of loss function is a **business decision** (see 9.5).

### 9.2 Reliability of the forecast intervals

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, name in zip(axes, ["ETS", "SARIMA", "Prophet"]):
    lo, hi = intervals[name]
    ax.fill_between(test.index, lo, hi, color=COLORS[name], alpha=0.25, label="95% CI")
    ax.plot(forecasts[name], color=COLORS[name])
    ax.scatter(test.index, test, color="k", s=15, zorder=3, label="Actual")
    cov = results[name]["95% CI coverage (%)"]
    width = np.mean(np.asarray(hi) - np.asarray(lo))
    ax.set_title(f"{name} — coverage {cov:.0f}%, avg. width {width:,.0f} GWh")
    ax.tick_params(axis="x", rotation=45)
axes[0].legend(); plt.tight_layout(); plt.show()

**Interpretation.**

- A **well-calibrated** 95% interval should contain about 11 to 12 of the 12 actual points. A noticeably lower coverage means the model is **overconfident**.
- Coverage and width must be read **together**: a very wide interval covers everything but is useless for decision-making. The best model is the one that reaches the nominal coverage with the narrowest interval.
- **ETS(A,A,M)**: 42% coverage with narrow intervals (≈ 500 GWh wide on average). It is **overconfident**, for the structural reason seen in 6.3 (additive variance).
- **SARIMA** and **Prophet**: 100% coverage. Prophet's intervals are narrower (≈ 920 GWh vs. ≈ 1,100 for SARIMA): this is the **best trade-off** here between coverage and width.
- Note: Prophet is often criticized for intervals that are too narrow, because it ignores error autocorrelation. Here, working in logs and the uncertainty on the trend (simulated from past changepoints) compensate for this. This result is specific to this series and should not be generalized.
- *Business.* For RTE, it is the **upper bound** of the interval that matters (sizing export capacity and curtailment in case of surplus). For a producer selling forward, it is the **lower bound** (not selling more than they will actually produce). Poorly calibrated intervals directly expose them to imbalance penalties or compensation purchases at spot price.

### 9.3 Rolling-origin cross-validation

A single test on 2025 is fragile, since 2025 is an atypical year. We therefore repeat the exercise **4 times**, moving the forecast origin forward. Each model is re-estimated on the data available at that date, then forecasts the following year:

| Fold | Training | Test |
|---|---|---|
| 1 | 2014–2021 | 2022 |
| 2 | 2014–2022 | 2023 |
| 3 | 2014–2023 | 2024 |
| 4 | 2014–2024 | 2025 |

The **specifications** (ETS(A,A,M), the SARIMA order, the Prophet configuration) stay fixed. Only the parameters are re-estimated.

In [ ]:
def run_all(tr, h=12):
    out = {"Seasonal naive": pd.Series(tr[-12:].values)}
    e = ETSModel(tr, error=best["error"], trend=best["trend"], damped_trend=best["damped"],
                 seasonal=best["seasonal"], seasonal_periods=12).fit(disp=False)
    out["ETS"] = pd.Series(e.forecast(h).values)
    s = SARIMAX(np.log(tr), order=sel["order"], seasonal_order=sel["sorder"],
                trend="c" if sel["order"][1] == 0 else "n").fit(disp=False)
    out["SARIMA"] = pd.Series(np.exp(s.get_forecast(h).predicted_mean).values)
    pm = fit_prophet(tr)
    out["Prophet"] = pd.Series(predict_prophet(pm, h)["yhat"].iloc[-h:].values)
    return out

cv_rows, cv_errors = [], {m: [] for m in COLORS if m != "Actual"}
for year in [2022, 2023, 2024, 2025]:
    tr, te = y[: str(year - 1)], y[str(year)]
    fcs = run_all(tr)
    for name, f in fcs.items():
        r = evaluate(te, f.values, tr); r.update({"Fold (test)": year, "Model": name})
        cv_rows.append(r); cv_errors[name].extend(list(te.values - f.values))

cv = pd.DataFrame(cv_rows)
cv.pivot(index="Fold (test)", columns="Model", values="MAPE (%)").round(1)

In [ ]:
cv_mean = cv.groupby("Model")[["RMSE", "MAE", "MAPE (%)", "MASE", "Bias (GWh)"]].mean().sort_values("RMSE")
display(cv_mean.style.format("{:,.2f}").highlight_min(subset=["RMSE", "MAE", "MAPE (%)", "MASE"], color="#c6efce"))

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
cv.pivot(index="Fold (test)", columns="Model", values="RMSE").plot(
    marker="o", ax=axes[0], color=[COLORS[c] for c in sorted(cv["Model"].unique())])
axes[0].set_title("RMSE by fold (GWh)"); axes[0].set_xticks([2022, 2023, 2024, 2025])
cv.pivot(index="Fold (test)", columns="Model", values="Bias (GWh)").plot(
    kind="bar", ax=axes[1], color=[COLORS[c] for c in sorted(cv["Model"].unique())])
axes[1].axhline(0, color="k", lw=0.8); axes[1].set_title("Average bias by fold (GWh, > 0 = under-forecast)")
plt.tight_layout(); plt.show()

**Interpretation.**

- Cross-validation gives a **more robust** picture than the single 2025 test. Averaged over 4 years, **Prophet** remains in the lead (RMSE ≈ 257 GWh, MAPE ≈ 11%), closely followed by **SARIMA** (≈ 276 GWh, ≈ 11%), then **ETS** (≈ 313 GWh, ≈ 13.5%). The naive method is far behind (≈ 466 GWh, ≈ 20%).
- No model wins **every** year: SARIMA is best in 2022 and 2024, Prophet in 2023 and 2025. This is a strong argument in favor of combining models.
- The **bias per fold** depends on the year:
  - acceleration years (2022: +35%; 2025: +33%) are **under-forecast by every model**;
  - in 2024, a year of weaker growth (+10%), ETS and Prophet **over-forecast**, because they had learned the strong 2022–2023 slope.

  Models **react with a lag** to changes in the deployment pace. Prophet has the lowest average bias (≈ 70 GWh vs. ≈ 200 for ETS and SARIMA).
- *Business.* This behavior is typical of models that extrapolate the past. In practice, an analyst should **adjust the trend** using the connection data published each quarter (SDES dashboard, Enedis data), rather than relying solely on the model.

### 9.4 Are the errors significantly different? The Diebold-Mariano test

The DM test compares the losses of two models: $d_t = L(e_{1,t}) - L(e_{2,t})$, with H₀: $E[d_t] = 0$ (equal accuracy). We use the quadratic loss (MSE) and the Harvey, Leybourne & Newbold (1997) small-sample correction.

⚠️ We stack the 48 errors from the 4 folds. They are autocorrelated within a fold, so the test is **indicative** and does not constitute formal proof.

In [ ]:
def dm_test(e1, e2, h=1, power=2):
    e1, e2 = np.asarray(e1), np.asarray(e2)
    d = np.abs(e1) ** power - np.abs(e2) ** power
    n = len(d); dbar = d.mean()
    gamma = [np.sum((d[k:] - dbar) * (d[:n - k] - dbar)) / n for k in range(h)]
    var = (gamma[0] + 2 * sum(gamma[1:])) / n
    dm = dbar / np.sqrt(var)
    hln = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    stat = dm * hln
    return stat, 2 * stats.t.sf(abs(stat), df=n - 1)

pairs = [("ETS", "SARIMA"), ("ETS", "Prophet"), ("SARIMA", "Prophet"),
         ("ETS", "Seasonal naive"), ("SARIMA", "Seasonal naive"), ("Prophet", "Seasonal naive")]
dm_rows = []
for a, b in pairs:
    s, pv = dm_test(cv_errors[a], cv_errors[b])
    dm_rows.append({"Comparison": f"{a} vs {b}", "DM stat": s, "p-value": pv,
                    "Conclusion (5%)": ("significant diff. — " + (a if s < 0 else b) + " better") if pv < 0.05 else "No significant difference"})
pd.DataFrame(dm_rows).set_index("Comparison").round(3)

**Interpretation.** A negative DM statistic means the first model has the lower loss.

- **All three models significantly beat the naive baseline** (p < 0.01).
- **Prophet is significantly better than ETS** (p ≈ 0.02).
- **SARIMA is not significantly distinguishable from either ETS or Prophet** (p ≈ 0.10 and ≈ 0.55).
- *Reading:* the advantage of Prophet over SARIMA seen in the tables is **not statistically established**. We can conclude that the log + trend structure (Prophet, SARIMA) does better than an additive trend in GWh (ETS(A,A,M)), but we cannot separate Prophet from SARIMA.
- *Business.* With only 4 years of test data, it would be unwise to definitively claim "Prophet is the best model." We can, however, decide on other grounds: interpretability of the changepoints, quality of the intervals, ease of maintenance.

### 9.5 Which loss function for which use?

| User | Stake | Suitable loss | Why |
|---|---|---|---|
| **RTE / grid security** | Unanticipated summer surplus, congestion | **RMSE / MSE**, or asymmetric loss | Large summer errors cost disproportionately (curtailment, forced export) |
| **Producer selling forward** | Selling more than actual production, having to buy back at spot price | **Quantile (pinball) loss** on a low quantile, e.g. 10% | Over-forecasts cost more than under-forecasts |
| **Finance department** | Annual revenue budget | **Error on the annual total**, MAE | Offsetting errors between months is acceptable |
| **Communication / reporting** | Readability | **MAPE** | A percentage figure that is easy to understand, but overweights winter (small volumes) |

MSE, the loss function the three models implicitly minimize when being estimated (via the Gaussian likelihood), is **not necessarily** the user's loss function. A model that is optimal in the MSE sense can be suboptimal for a decision whose costs are asymmetric.

## 10. 2026 Forecast

We re-estimate the three models on the **entire 2014–2025 series** and forecast the 12 months of 2026. We add a **combination** (simple average of the three forecasts). The literature shows that it is often more robust than any single model (Makridakis et al., M4/M5 competitions).

In [ ]:
fut_idx = pd.date_range("2026-01-01", periods=12, freq="MS")

ets_all = ETSModel(y, error=best["error"], trend=best["trend"], damped_trend=best["damped"],
                   seasonal=best["seasonal"], seasonal_periods=12).fit(disp=False)
p_ets = ets_all.get_prediction(start=fut_idx[0], end=fut_idx[-1]).summary_frame(alpha=0.05)

sar_all = SARIMAX(np.log(y), order=sel["order"], seasonal_order=sel["sorder"],
                  trend="c" if sel["order"][1] == 0 else "n").fit(disp=False)
f_sar = sar_all.get_forecast(12); ci_sar = np.exp(f_sar.conf_int(alpha=0.05))

pr_all = fit_prophet(y); f_pr = predict_prophet(pr_all, 12).loc[fut_idx]

fc26 = pd.DataFrame({"ETS": p_ets["mean"].values,
                     "SARIMA": np.exp(f_sar.predicted_mean).values,
                     "Prophet": f_pr["yhat"].values}, index=fut_idx)
fc26["Combination"] = fc26.mean(axis=1)
fc26["95% CI low (min)"] = np.minimum.reduce([p_ets["pi_lower"].values, ci_sar.iloc[:, 0].values, f_pr["yhat_lower"].values])
fc26["95% CI high (max)"] = np.maximum.reduce([p_ets["pi_upper"].values, ci_sar.iloc[:, 1].values, f_pr["yhat_upper"].values])

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y["2022":], color="k", lw=2, label="History")
for c in ["ETS", "SARIMA", "Prophet"]:
    ax.plot(fc26[c], color=COLORS[c], lw=1.2, alpha=0.7, label=c)
ax.plot(fc26["Combination"], color="crimson", lw=2.5, label="Combination")
ax.fill_between(fut_idx, fc26["95% CI low (min)"], fc26["95% CI high (max)"], color="crimson", alpha=0.12,
                label="95% CI envelope")
ax.set_title("Solar production forecast 2026 (GWh)"); ax.legend(ncol=3); plt.show()

out = fc26.copy(); out.index = out.index.strftime("%Y-%m")
out.loc["Total 2026"] = out.sum()
print(f"Actual 2025 production: {y['2025'].sum()/1000:.1f} TWh")
print(f"Forecast 2026 production (combination): {fc26['Combination'].sum()/1000:.1f} TWh "
      f"({(fc26['Combination'].sum()/y['2025'].sum()-1)*100:+.1f}%)")
out.round(0)

**Interpretation.**

- *Technical.* Re-estimated with 2025 included, the three models forecast 2026 production between **36.6 TWh** (SARIMA) and **38.8 TWh** (ETS). The combination gives **≈ 37.9 TWh, i.e. +15%** relative to 2025 (32.9 TWh). The **envelope** of the 95% intervals (minimum of the lower bounds, maximum of the upper bounds) spans roughly **29 to 46 TWh** over the year. This is a conservative measure of total uncertainty, which includes uncertainty about model choice.
- The models therefore anticipate growth **twice as weak as in 2025** (+15% vs. +33%). They "average out" history and treat 2025 as partly exceptional. If the 2025 installation pace is sustained, they will likely underestimate 2026, as they underestimated 2025.
- *Business.* The combined forecast gives an order of magnitude for expected solar production in 2026 and its monthly breakdown. The months of **May to August 2026** should set new monthly records (July ≈ 4.9 TWh vs. 4.5 TWh in July 2025). This is key information for:
  - **RTE**: anticipating more frequent surplus episodes around midday and the need for export, storage and demand-side management (EV charging, water heaters shifted to midday);
  - **EDF**: concentrating nuclear maintenance outages in spring and summer;
  - **producers**: expecting **cannibalization** of market prices during solar hours. The more solar there is, the lower the price captured by solar becomes. This phenomenon does not appear in a volume model, but it drives profitability.
- ⚠️ These forecasts **do not incorporate any information about future grid connections**. If the 2025 installation pace is sustained or accelerates, the models will likely remain below reality, as they did in 2025.

## 11. Business Recommendations

1. **Use Prophet (log) as the primary model, SARIMA as a challenger, and publish the combination as the reference estimate.** Prophet is the most accurate on average and its changepoints are easy to explain. But its edge over SARIMA is not statistically established, and no model wins every year. Averaging reduces the risk of betting on the wrong model in the year it falters.
2. **Correct the structural under-forecasting bias** by adding **installed capacity** as an explanatory variable: SARIMAX with an exogenous regressor, or `add_regressor` in Prophet. Connected-capacity data is published quarterly (SDES, ODRÉ). Modeling the **capacity factor** instead (production / capacity), which is stationary and purely weather-driven, would cleanly separate the two sources of uncertainty.
3. **Revise the forecast every month** (*rolling forecast*) rather than once a year. Uncertainty grows sharply with the horizon, and models react with a lag to changes in pace.
4. **Choose the loss function according to the use case** (section 9.5). In particular, **quantile forecasts** are needed for decisions with asymmetric costs (forward sales, grid sizing).
5. **Prepare for summer**: multiplicative seasonality means each installed GW adds roughly 4 times more production in July than in December. Summer flexibility (storage, demand response, export) must grow **at the same pace as the fleet**, not at the pace of annual production.

## 12. Conclusion and Limitations

### Conclusion
We modeled French monthly solar production (2014–2025) using three families of methods. Several conclusions converge:

- The series is dominated by an **exponential trend** (fleet deployment) and a **deterministic multiplicative seasonality** (astronomy). STL decomposition attributes more than 95% of the variance to these two components.
- The three methods **independently** recovered these properties, which reinforces confidence in the diagnosis:
  - **ETS** selects multiplicative seasonality with γ ≈ 0;
  - **SARIMA** estimates a seasonal MA ≈ −1 (a sign of deterministic seasonality) and highly persistent growth (AR ≈ 0.93);
  - **Prophet** locates growth-regime changes consistent with the sector's history.
- All three models **clearly beat** the naive method (MAPE roughly halved). **Prophet on the log** is the most accurate, both in the 2025 test (MAPE 8.5%) and in cross-validation. It significantly beats ETS, but not SARIMA.
- The quality of the **intervals** varies a lot: ETS with additive error is overconfident (42% coverage for a stated 95%), whereas SARIMA and Prophet, fitted on the log, are reliable. The logarithmic transform was the single most decisive modeling choice in this project.
- Their common limitation is an **under-forecasting bias during deployment accelerations** (2022, 2025). A univariate model cannot anticipate a shift in pace that is not present in the historical record.

### Limitations
- **Short sample**: 144 months, i.e. only 12 seasonal cycles and 4 validation folds. Statistical tests therefore have limited power.
- **No exogenous variables**: installed capacity, sunshine, prices.
- **National aggregation**: it hides regional heterogeneity and local grid constraints.
- **Monthly granularity**: the operational issues (negative prices, curtailment) are **hourly**. An analysis on half-hourly éCO2mix data would be the next step.
- **2025 data possibly provisional** — **verified**: the dataset's description page confirms that this type of revision has already happened (2022-2024 solar production revised upward after a recalculation of self-consumption, updated 25/02/2026). Nothing guarantees that 2025 will not be revised the same way. See the detail in §2.
- **Self-consumption**: self-consumed production is not precisely measured by the grid (only grid-fed electricity is). **Quantified**: our 2025 total (32.9 TWh, grid-fed) is about **4 TWh (11%) below** the total of the [SDES Q4 2025 dashboard](https://www.statistiques.developpement-durable.gouv.fr/tableau-de-bord-solaire-photovoltaique-quatrieme-trimestre-2025) (37.0 TWh), which incorporates an estimate of self-consumption. Our forecasts therefore structurally underestimate the fleet's **total** production, beyond the growth under-forecasting bias already identified above. See the detail in §2.

## 13. References

### Data and sector context
- **Data:** ODRÉ / RTE, *Monthly regional electricity production by source*, [data.gouv.fr](https://www.data.gouv.fr/datasets/production-regionale-mensuelle-delectricite-par-filiere) (last updated: 25 February 2026; mentions a recalculation of photovoltaic self-consumption that raised 2022-2024 production).
- RTE (2026). *[2025 Electricity Report — key results](https://assets.rte-france.com/prod/public/2026-02/Bilan-electrique-2025-principaux-resultats.pdf)*. National solar production 2025, annual growth, installed capacity, hours of negative prices and solar/wind curtailment.
- SDES (2026). *[Dashboard: solar photovoltaic — Fourth quarter 2025](https://www.statistiques.developpement-durable.gouv.fr/tableau-de-bord-solaire-photovoltaique-quatrieme-trimestre-2025)*. Installed capacity and national production 2025, including an estimate of self-consumption.
- SDES (2022). *[Methodological note — Production and capacity of the photovoltaic fleet](https://www.statistiques.developpement-durable.gouv.fr/sites/default/files/2022-02/methodologie_pv_fevrier2022.pdf)*. Method for estimating photovoltaic self-consumption from 2020 onward.
- Techniques de l'Ingénieur (18 February 2026). *["The multi-year energy plan finally arrives, by decree"](https://www.techniques-ingenieur.fr/actualite/articles/energies-la-programmation-arrive-enfin-par-decret-155776/amp)*. PPE3 targets: solar photovoltaic capacity of 48 GW in 2030 and 55-80 GW in 2035 (decree presented on 13 February 2026).

### Methods
- Hyndman, R.J. & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice*, 3rd ed., OTexts. https://otexts.com/fpp3/
- Hyndman, R.J., Koehler, A.B., Ord, J.K. & Snyder, R.D. (2008). *Forecasting with Exponential Smoothing: The State Space Approach*. Springer.
- Box, G.E.P., Jenkins, G.M., Reinsel, G.C. & Ljung, G.M. (2015). *Time Series Analysis: Forecasting and Control*, 5th ed. Wiley.
- Taylor, S.J. & Letham, B. (2018). Forecasting at scale. *The American Statistician*, 72(1), 37–45.
- Cleveland, R.B., Cleveland, W.S., McRae, J.E. & Terpenning, I. (1990). STL: A seasonal-trend decomposition procedure based on loess. *Journal of Official Statistics*, 6(1), 3–73.
- Wang, X., Smith, K. & Hyndman, R.J. (2006). Characteristic-based clustering for time series data. *Data Mining and Knowledge Discovery*, 13, 335–364.
- Diebold, F.X. & Mariano, R.S. (1995). Comparing predictive accuracy. *Journal of Business & Economic Statistics*, 13(3), 253–263.
- Harvey, D., Leybourne, S. & Newbold, P. (1997). Testing the equality of prediction mean squared errors. *International Journal of Forecasting*, 13(2), 281–291.
- Makridakis, S., Spiliotis, E. & Assimakopoulos, V. (2020). The M4 Competition: 100,000 time series and 61 forecasting methods. *International Journal of Forecasting*, 36(1), 54–74.

*All URLs were accessed on 15 October 2026.*

## 14. Course Glossary — where each concept appears in this notebook

| Concept | Short definition | Where it is applied here |
|---|---|---|
| **Additive / multiplicative decomposition** | $Y_t=T_t+S_t+R_t$ vs $Y_t=T_t\times S_t\times R_t$ | §4.1 — direct comparison, multiplicative chosen based on the residual funnel |
| **CMA** (*Centered Moving Average*) | A centered moving average used to isolate $T_t$ by smoothing out $S_t+R_t$ | Internal mechanism of `seasonal_decompose` (§4.1), limitation spelled out in §4.2 |
| **X-11 / X-13ARIMA-SEATS** | Official seasonal-adjustment procedures (Insee, Eurostat), iterative filters + ARIMA extrapolation | §4.2 — not used, STL preferred, choice justified |
| **STL** | Seasonal-trend decomposition by LOESS, robust to outliers | §4.2 — applied to $\log y_t$ |
| **Stationarity** | Mean, variance and autocovariance constant over time | §7.1 — required condition before fitting a SARIMA |
| **Unit Root / Unit Root Test** | H₀: the series follows a random walk (non-stationary) | §7.1 — ADF test |
| **ADF test** | H₀ = unit root; p < 0.05 ⟹ stationary | §7.1, cross-checked with KPSS |
| **KPSS test** | H₀ = stationary; p < 0.05 ⟹ non-stationary | §7.1, hypotheses opposite to ADF to guard against false positives |
| **Stochastic trend** | Random walk with drift; a shock has a permanent effect; removed by **differencing** | §7.1 — the case of log(y), confirmed by the non-rejected unit root |
| **Deterministic trend** | A fixed function of time ($\alpha+\beta t$); a shock fades, the series returns to its trajectory; removed by regression on $t$ | §7.1 — mentioned by contrast, ruled out here |
| **White noise** | A residual with zero mean, constant variance, no autocorrelation | §7.4 — verified on the SARIMA residuals via Ljung-Box |
| **ACF / PACF** | Autocorrelation functions (simple / partial), used to identify $p$ and $q$ | §7.2 — on $\Delta_{12}\log y$, before the grid search |
| **AR(p) model** | $Y_t$ depends linearly on its $p$ past values | §7.3 — AR(1) coefficient ≈ 0.93 interpreted |
| **ETS (Error-Trend-Seasonal)** | State-space exponential smoothing, with AIC and forecast intervals | §6 — AIC selection, α/β/γ parameters interpreted |
| **Damped trend** | ETS variant where the slope slows toward a plateau | §6.1 — tested and ruled out by AIC |
| **SARIMA** | ARIMA + seasonal component $(P,D,Q)_s$ | §7 — identified via ACF/PACF then AIC grid search |
| **Prophet** | $y_t=g(t)+s(t)+h(t)+\varepsilon_t$; piecewise-linear trend + Fourier seasonality + changepoints | §8 |
| **Changepoints** | Dates where the slope of the Prophet trend is allowed to change | §8.1-8.2 |
| **MSE / RMSE** | Squared error (mean / root); heavily penalizes large errors | §5.2, §9.1 |
| **MAE** | Mean absolute error, robust to outliers | §5.2, §9.1 |
| **MAPE** | Mean relative error, in % | §5.2, §9.1 |
| **MASE** | MAE scaled by the seasonal naive's MAE; unitless, < 1 = better than the naive method | §5.2, §9.1 |
| **Rolling-origin cross-validation** | Repeating the train/test split while moving the origin forward in time | §9.3 |
| **Diebold-Mariano test** | Tests whether two models have significantly different losses | §9.4 |
| **Forecast combination** | Average of several models, often more robust than any single one | §10 |

*This glossary is an index, not a substitute: each row points to the place in the notebook where the concept is both computed **and** interpreted (technically and in business terms), as required by the project brief.*